# A FAIRE POUR LANCER UN GROS ENTRAINEMENT

- utiliser un modèle b3 (ou +) que b0
- en csq changer le target size dans le preprocessor
- augmenter nombre d'epoch
- batch size ?

Optionnel :
- tester une autre loss

# **0. Librairies**

In [2]:
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader
from tqdm import tqdm
import sys
import os


sys.path.append(os.path.abspath(".."))

from lib.data.dataset import BeeDataset

from lib.data.preprocessing import TorchPreprocessor

from lib.data.train_val_split import train_val_split

from lib.data.preprocessing import TorchPreprocessor

from lib.data.data_augmentation import data_augmented_loader

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 32
EPOCHS = 25
LR = 1e-4
num_classes = 50

notebook_dir = os.getcwd()

data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "data"))

## **1. Preprocessing**

In [4]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# preprocessor = TorchPreprocessor(
#     normalize=True,
#     mean = IMAGENET_MEAN,
#     std = IMAGENET_STD,
#     resize_method="pad",
#     target_size=(224, 224),
# )

# train_dataset, val_dataset = train_val_split(train_transform=preprocessor, val_transform=preprocessor)


In [5]:
import lib.data.preprocessing as prep
print(prep.__file__)

/home/audrey-maurette/Documents/ISAE-Supaero/SDD/Hackaton_abeilles_tigres/lib/data/preprocessing.py


In [6]:
heavy_training_preprocessor = TorchPreprocessor(
    normalize=True,
    mean = IMAGENET_MEAN,
    std = IMAGENET_STD,
    augmentation="heavy",
    resize_method="pad",
    target_size=(224, 224),
    interpolation_method="BICUBIC",
)

light_training_preprocessor = TorchPreprocessor(
    normalize=True,
    mean = IMAGENET_MEAN,
    std = IMAGENET_STD,
    augmentation="light",
    resize_method="pad",
    target_size=(224, 224),
    interpolation_method="BICUBIC",
)

val_preprocessor = TorchPreprocessor(
    normalize=True,
    mean = IMAGENET_MEAN,
    std = IMAGENET_STD,
    augmentation="none",
    resize_method="pad",
    target_size=(224, 224),
    interpolation_method="BICUBIC",
)


train_loader, val_loader = data_augmented_loader(IMAGENET_MEAN, IMAGENET_STD, target_size=(224, 224), batch_size=BATCH_SIZE, train_preprocessor_light=light_training_preprocessor, train_preprocessor_heavy=heavy_training_preprocessor, val_preprocessor=val_preprocessor)

Train prêt : 6417 images (avec augmentation ciblée)
Val prête  : 1582 images (sans augmentation)


In [13]:
import pandas as pd

# --- Récupérer tous les labels du val_loader ---
all_labels = []

for _, labels in val_loader:
    all_labels.extend(labels.numpy() if isinstance(labels, torch.Tensor) else labels)

# --- Compter le nombre d'images par classe ---
class_counts_val = pd.Series(all_labels).value_counts().sort_index()

# --- Créer un DataFrame ---
df_val_counts = pd.DataFrame({
    "class": class_counts_val.index,
    "num_images": class_counts_val.values
})

# --- Sauvegarder dans un CSV ---
csv_val_path = "val_class_counts.csv"
df_val_counts.to_csv(csv_val_path, index=False)
print(f"CSV des images par classe dans la validation sauvegardé dans : {csv_val_path}")

CSV des images par classe dans la validation sauvegardé dans : val_class_counts.csv


## **2. Modèle**

In [7]:
from torch.optim.lr_scheduler import CosineAnnealingLR

def create_model(num_classes: int) -> nn.Module:

    model = models.efficientnet_b0(weights="IMAGENET1K_V1") #mettre b3 si ca marche
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, num_classes),
    )
    return model

model = create_model(num_classes)
model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# --- Variante ---
# pip install torchmetrics
# from torchmetrics.classification import MulticlassFocalLoss
# criterion = MulticlassFocalLoss(num_classes=num_classes, alpha=0.25, gamma=2.0)

# FocalLoss peut être redondante avec l'utilisation d'un WeightedRandomSampler
# Label Smoothing dans la CrossEntropyLoss ? A 0.1 par exemple

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

# scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# Usage de OneCycleLR
steps_per_epoch = len(train_loader)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    pct_start=0.1, # 10% du temps d'entraînement sera dédié au Warm-up (montée douce du LR)
    div_factor=10.0, # Le LR commence à LR / 10
    final_div_factor=100.0 # Le LR finit très bas
)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /home/oclaich/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:16<00:00, 1.32MB/s]


## **3. F1-score**

In [8]:
import numpy as np

def compute_f1(all_labels, all_preds, num_classes):
    f1_per_class = []

    for cls in range(num_classes):
        # True Positives
        TP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) == cls))
        # False Positives
        FP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) != cls))
        # False Negatives
        FN = np.sum((np.array(all_preds) != cls) & (np.array(all_labels) == cls))

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0

        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_per_class.append(f1)

    # F1 macro : moyenne des classes
    f1_macro = np.mean(f1_per_class)
    return f1_macro, f1_per_class

## **4. Fonctions de training et validation**

In [9]:
def train_one_epoch():
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    for images, labels in tqdm(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    # 🔹 Calcul F1 avec ta fonction existante
    f1_macro, f1_per_class = compute_f1(all_labels, all_preds, num_classes)

    # 🔹 Affichage
    # print(f"Train F1 macro: {f1_macro:.4f}")

    return total_loss / len(train_loader), correct / total, f1_macro, f1_per_class


In [10]:
import torch

def validate():
    model.eval()
    total_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calcul F1 manuel par classe
    f1_per_class = []
    for cls in range(num_classes):
        TP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) == cls))
        FP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) != cls))
        FN = np.sum((np.array(all_preds) != cls) & (np.array(all_labels) == cls))
        
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
        
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_per_class.append(f1)
    
    f1_macro = np.mean(f1_per_class)

    return (total_loss / len(val_loader), f1_macro, f1_per_class, np.array(all_preds), np.array(all_labels))

## **5. Entrainement**

**Vérif des labels**

In [11]:
# all_labels = [label for _, label in train_dataset]
# print("Label min:", min(all_labels))
# print("Label max:", max(all_labels))
# print("Nombre de classes uniques:", len(set(all_labels)))

# # Récupérer tous les labels uniques triés
# all_labels = sorted(set(label for _, label in train_dataset.samples))
# label_to_index = {label: i for i, label in enumerate(all_labels)}

# # Remapper les labels dans le dataset
# # for i, (path, label) in enumerate(train_dataset.samples):
# #     train_dataset.samples[i] = (path, label_to_index[label])

**Entrainement**

In [12]:
import csv
import pandas as pd
from sklearn.metrics import confusion_matrix

best_f1 = 0.0
best_model_path = "best_model.pth"

csv_file = "training_log.csv"
fieldnames = ['epoch', 'train_loss', 'train_acc', 'train_f1_macro',
              'val_loss', 'val_f1_macro']

# Initialisation du CSV
with open(csv_file, mode='w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

for epoch in range(EPOCHS):

    # ===== TRAIN =====
    train_loss, train_acc, train_f1_macro, train_f1_per_class = train_one_epoch()
    scheduler.step()

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Train F1 Macro: {train_f1_macro:.4f}")

    # ===== VALIDATION =====
    val_loss, val_f1_macro, val_f1_per_class, val_preds, val_labels = validate()

    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val F1 Macro: {val_f1_macro:.4f}")

    # Log CSV
    with open(csv_file, mode='a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow({
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'train_f1_macro': train_f1_macro,
            'val_loss': val_loss,
            'val_f1_macro': val_f1_macro
        })

    # ===== BEST MODEL =====
    if val_f1_macro > best_f1:

        best_f1 = val_f1_macro
        torch.save(model.state_dict(), best_model_path)

        print(f" Nouveau meilleur modèle sauvegardé ! F1 Macro = {best_f1:.4f}")

        # =============================
        # CONFUSION MATRIX
        # =============================
        cm = confusion_matrix(val_labels, val_preds)
        cm_df = pd.DataFrame(cm)
        cm_df.to_csv("best_model_confusion_matrix.csv", index=False)

        # =============================
        # METRICS PAR CLASSE
        # =============================
        precision_per_class = []
        recall_per_class = []
        f1_per_class = []
        accuracy_per_class = []

        for cls in range(num_classes):

            TP = cm[cls, cls]
            FP = cm[:, cls].sum() - TP
            FN = cm[cls, :].sum() - TP
            TN = cm.sum() - (TP + FP + FN)

            precision = TP / (TP + FP) if (TP + FP) > 0 else 0
            recall = TP / (TP + FN) if (TP + FN) > 0 else 0
            f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            accuracy_cls = (TP + TN) / cm.sum()

            precision_per_class.append(precision)
            recall_per_class.append(recall)
            f1_per_class.append(f1)
            accuracy_per_class.append(accuracy_cls)

        metrics_df = pd.DataFrame({
            "class": list(range(num_classes)),
            "accuracy": accuracy_per_class,
            "precision": precision_per_class,
            "recall": recall_per_class,
            "f1_score": f1_per_class
        })

        metrics_df.to_csv("best_model_metrics_per_class.csv", index=False)

        print(" Metrics par classe sauvegardées")

  0%|          | 0/201 [00:00<?, ?it/s]

100%|██████████| 201/201 [11:26<00:00,  3.42s/it]



Epoch 1/25
Train Loss: 3.2275 | Train Acc: 0.2906
Train F1 Macro: 0.2685
Val Loss: 2.7450
Val F1 Macro: 0.2471
 Nouveau meilleur modèle sauvegardé ! F1 Macro = 0.2471
 Metrics par classe sauvegardées


100%|██████████| 201/201 [12:31<00:00,  3.74s/it]



Epoch 2/25
Train Loss: 1.7220 | Train Acc: 0.6241
Train F1 Macro: 0.5963
Val Loss: 1.7275
Val F1 Macro: 0.3855
 Nouveau meilleur modèle sauvegardé ! F1 Macro = 0.3855
 Metrics par classe sauvegardées


...

100%|██████████| 201/201 [13:02<00:00,  3.89s/it]


Epoch 5/25
Train Loss: 0.5373 | Train Acc: 0.8579
Train F1 Macro: 0.8559


Val Loss: 1.0314
Val F1 Macro: 0.5362
 Nouveau meilleur modèle sauvegardé ! F1 Macro = 0.5362
 Metrics par classe sauvegardées


  3%|▎         | 7/201 [00:24<11:10,  3.45s/it]


KeyboardInterrupt: 

## **6. Création du fichier submission**

In [ ]:
from torch.utils.data import DataLoader
import pandas as pd
import torch
from lib.data.dataset import BeeDataset

def submission(model, batch_size=32, transform=None, model_path="best_model.pth", save_path="submission.csv"):
    # Charger le modèle sur le bon device
    model.load_state_dict(torch.load(model_path, map_location=torch.device(DEVICE)))
    model.to(DEVICE)
    model.eval()
    
    # Dataset de test
    test_dataset = BeeDataset(train=False, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    all_ids = []
    all_preds = []
    
    with torch.no_grad():
        for imgs, ids in test_loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            
            preds = torch.argmax(outputs, dim=1)
            
            # Convertir preds en int et ids en int ou str
            all_preds.extend(preds.cpu().tolist())
            all_ids.extend([int(x) if isinstance(x, torch.Tensor) else x for x in ids])
    
    submission_df = pd.DataFrame({
        "id": all_ids,
        "label": all_preds
    })
    
    submission_df.to_csv(save_path, index=False)
    print(f"Submission saved to {save_path}")

In [ ]:
test_dataset = BeeDataset(train=False, transform=val_preprocessor)

model_path = "/home/alexandre-tonon/SDD/Hackathons/Hackaton_abeilles_tigres/models/efficientnetb0_audrey_better_loader_25epoch.pth"
submission(model, batch_size=32, transform=val_preprocessor, model_path=model_path, save_path="submission.csv")

/tmp/ipykernel_10224/2135808320.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=torch.device(DEVICE)))


Submission saved to submission.csv


: 

## **8. Création des fichiers résultats à partir d'un modèle enregistré dans un .pth**

In [ ]:
from torch.utils.data import DataLoader
import pandas as pd
import torch
from lib.data.dataset import BeeDataset
from lib.data.preprocessing import TorchPreprocessor
from sklearn.metrics import confusion_matrix

model_path = os.path.join("..", "models", "efficientnetb0_audrey_better_loader_25epoch.pth")

def evaluate_model(model, batch_size=32, model_path=model_path, transform=None, num_classes=None):
    """
    Évalue un modèle sur le dataset de validation.
    
    Génère :
    - best_model_confusion_matrix.csv
    - best_model_metrics_per_class.csv
    """
    # Charger le modèle
    model.load_state_dict(torch.load(model_path, map_location=torch.device(DEVICE)))
    model.to(DEVICE)
    model.eval()
    
    # Dataset de validation
    # val_dataset = BeeDataset(train=False, transform=transform)
    # val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    
    # Détecter num_classes si pas donné
    if num_classes is None:
        num_classes = max(max(all_labels), max(all_preds)) + 1
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    cm_df = pd.DataFrame(cm)
    cm_df.to_csv("best_model_confusion_matrix_bis.csv", index=False)
    print("Confusion matrix saved to best_model_confusion_matrix.csv")
    
    # Metrics par classe
    precision_per_class = []
    recall_per_class = []
    f1_per_class = []
    accuracy_per_class = []
    
    for cls in range(num_classes):
        TP = cm[cls, cls]
        FP = cm[:, cls].sum() - TP
        FN = cm[cls, :].sum() - TP
        TN = cm.sum() - (TP + FP + FN)
        
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        accuracy_cls = (TP + TN) / cm.sum()
        
        precision_per_class.append(precision)
        recall_per_class.append(recall)
        f1_per_class.append(f1)
        accuracy_per_class.append(accuracy_cls)
    
    metrics_df = pd.DataFrame({
        "class": list(range(num_classes)),
        "accuracy": accuracy_per_class,
        "precision": precision_per_class,
        "recall": recall_per_class,
        "f1_score": f1_per_class
    })
    
    metrics_df.to_csv("best_model_metrics_per_class_bis.csv", index=False)
    print("Metrics per class saved to best_model_metrics_per_class.csv")
    
    return cm_df, metrics_df

In [18]:
evaluate_model(model, batch_size=32, transform=val_preprocessor, model_path=model_path, num_classes=num_classes)

Confusion matrix saved to best_model_confusion_matrix.csv
Metrics per class saved to best_model_metrics_per_class.csv


(      0     1     2     3     4     5     6     7     8     9     ...  1987  \
 0        0     0     0     0     0     0     0     0     0     0  ...     0   
 1        0     0     0     0     0     0     0     0     0     0  ...     0   
 2        0     0     0     0     0     0     0     0     0     0  ...     0   
 3        0     0     0     0     0     0     0     0     0     0  ...     0   
 4        0     0     0     0     0     0     0     0     0     0  ...     0   
 ...    ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  ...   ...   
 1992     0     0     0     0     0     0     0     0     0     0  ...     0   
 1993     0     0     0     0     0     0     0     0     0     0  ...     0   
 1994     0     0     0     0     0     0     0     0     0     0  ...     0   
 1995     0     0     0     0     0     0     0     0     0     0  ...     0   
 1996     0     0     0     0     0     0     0     0     0     0  ...     0   
 
       1988  1989  1990  1991  1992  1